# Describing Membranes As Meshes

## Overview

### Questions

* What is the Triangulated Mesh Model?
* How can I create a mesh modeling a flat surface?
* How can I create a mesh modeling a sphere?

### Objectives

- Introducing the Triangulated Mesh Model.
- Create a triangulation of the xy-plane.
- Create a triangulation of a sphere surface.
## Boilerplate Code

In [1]:
import numpy as np
import pyvista as pv

## Triangulated Mesh Model
Modeling membranes and other deformable interfaces as triangulated meshes is a common coarse-grained approach in molecular simulations, particularly for studying lipid bilayers and biological membranes at macroscopic scales.
The **triangulated mesh model** represents the interface surface as a set of vertices connected by edges forming a mesh of triangles.
Each vertex represents a coarse-grained membrane patch (e.g., a group of lipids).
This approach allows users to integrate membranes with specific elasticities and interactions with other particles.

![Triangulation](triangulation.svg)

## Creating a triangulation of the xy-plane.
To create a triangulation of the xy-plane the positions of the vertices have to defined first.

In [3]:
# box size
L = 10

# Number of vertices per row
row = 10

# Number of vertices
N = row*row

# Create meshgrid for a rectangular pattern
x = np.arange(-L/2, L/2, L/row)
y = np.arange(-L/2, L/2, L/row)*np.sqrt(3)/2
X, Y = np.meshgrid(x, y)

# Offset every second row to create a triangular pattern
X[1::2, :] += 0.5

# Combine X and Y coordinates to define vertex positions
vertex_positions = np.column_stack([X.flatten(), Y.flatten()])


In adition to the vertex positions also the triangulation data is needed.
The mesh information is given by an array of triplets which define each triangle in the mesh and consist of the vertex indices.

In [3]:
# Create array of vertex indices
v_index = np.arange(N)

# Find index of vertex to the right in same row
vr_index = v_index+1

# Consider periodic boundaries
vr_index[row-1::row] -=row

# Find index of vertex above and below the edge between idx1 and idx2
e_index = np.repeat(v_index,2).reshape(-1,2)

# Find index of vertex above the edge between v_index and vr_index
e_index[:,0]+=row
# Find index of vertex below the edge between v_index and vr_index
e_index[:,1]-=row

# Consider periodic boundaries
e_index = e_index.reshape(row,-1,2)
# periodic boundaries in x direction
e_index[1::2] += 1
# periodic boundaries in y direction
e_index[1::2,-1] -= row
e_index = e_index.reshape(-1)%N

# Create dublicates of the vertices to create triangle above and below each edge
v_index = np.repeat(v_index,2)
vr_index = np.repeat(vr_index,2)

# Switch vr_index and e_index for triangle below each edge to make mesh orientable
vr_index[1::2]=e_index[1::2]
e_index[1::2]=vr_index[0::2]

# Create all triangles making up the mesh
triangles = np.column_stack([v_index,vr_index,e_index])


## Creating a triangulation of a sphere surface.

First, we distribute vertices on a sphere using the fibancci algorithm (https://doi.org/10.1007/s11004-009-9257-x)

In [3]:
# Number of points on the sphere
num_pts = 1000

# Radius of the sphere
R = 10

#Fibancci algorithm
indices = arange(0, num_pts, dtype=float) + 0.5

phi = arccos(1 - 2*indices/num_pts)
theta = np.pi * (1 + 5**0.5) * indices

x, y, z = R * np.cos(theta) * np.sin(phi), R * np.sin(theta) * np.sin(phi), R * np.cos(phi)

vertex_positions = np.column_stack((x,y,z))

We determine the triangled by creating the convex hull on the vertex positions

In [3]:
mesh = pv.PolyData(vertex_positions)

# Delaunay triangulation and surface extraction:
triangulated = mesh.delaunay_3d()
surface = triangulated.extract_geometry()

faces = surface.faces.reshape((-1, 4))
# Remove first entry that states the number of vertices of the faces 
triangle_points = face[1:]

triangles = np.row_stack((triangle_points))

# Visualize the sphere mesh 
plotter = pv.Plotter()
plotter.add_mesh(surface, color='white', show_edges=True)
plotter.add_points(mesh_positions, color='red', point_size=5)
plotter.show()